# BÀI 2. XẾP HẬU (XEPHAU.*)

## Mô tả bài toán:
Xếp bàn cờ tổng quát kích thước N×N. Một quân hậu trên bàn cờ có thể ăn được các quân khác nằm tại các ô cùng hàng, cùng cột hoặc cùng đường chéo. 

Hãy liệt kê tất cả các cách xếp N quân hậu trên bàn cờ N×N sao cho không có quân hậu nào ăn được quân hậu khác.
(**3≤N≤100**) *(Lâu hơn cách: để dự quay lui và tham lam từng phần)*

## Dữ liệu:
**Nhập từ bàn phím:**
- Vào từ bàn phím chứa số nguyên N với N là kích thước cạnh của bàn cờ N×N

## Dữ liệu ra:
**Xuất ra màn hình:**
- Dòng đầu tiên ghi số 0 hoặc 1 (0 là không có cách xếp và 1 là có các cách xếp)
- Nếu có, N dòng tiếp theo, mỗi dòng có N số cách nhau bởi một khoảng trắng biểu diễn bàn cờ và cách xếp hậu. Nếu giá trị số 0 nghĩa là không có hậu. Ngược lại là 1 nghĩa là có hậu ở trí đó.

## Ví dụ:

**Bàn phím:** 5  
**Màn hình:** (có 1 lời giải)
```
1
1 0 0 0 0
0 0 1 0 0
0 0 0 0 1
0 1 0 0 0
0 0 0 1 0
```

**Bàn phím:** 2  
**Màn hình:** 0 *(không có lời giải)*

## Thuật toán 1: Quay lui (Backtracking)

### Ý tưởng:
- Duyệt từng hàng, thử đặt hậu vào các cột an toàn
- Dùng 3 mảng đánh dấu (cột, chéo chính, chéo phụ) để kiểm tra xung đột O(1)
- Khi tìm được 1 nghiệm, trả về ngay (tìm lời giải duy nhất)

In [1]:
def quay_lui(n):
    """
    Giải bài toán N-Queens sử dụng thuật toán Quay lui (Backtracking).
    Trả về bàn cờ nếu có lời giải, None nếu không có.
    """
    # Trường hợp đặc biệt
    if n == 2 or n == 3:
        return None

    board = [[0] * n for _ in range(n)]
    cot = [False] * n                      # Đánh dấu cột đã có hậu
    cheo_chinh = [False] * (2 * n - 1)    # Đánh dấu đường chéo chính
    cheo_phu = [False] * (2 * n - 1)      # Đánh dấu đường chéo phụ

    def Try(hang):
        # Base case: Đã xếp thành công N con hậu (hàng = N)
        if hang == n:
            return True
        
        # Thử đặt hậu vào từng cột của hàng hiện tại
        for c in range(n):
            # Kiểm tra ô (hang, c) có bị khống chế không
            if (not cot[c] and 
                not cheo_chinh[hang - c + n - 1] and 
                not cheo_phu[hang + c]):
                
                # Đặt hậu và đánh dấu
                board[hang][c] = 1
                cot[c] = True
                cheo_chinh[hang - c + n - 1] = True
                cheo_phu[hang + c] = True
                
                # Đi sâu vào hàng tiếp theo
                if Try(hang + 1):
                    return True
                
                # Backtrack: Phục hồi trạng thái nếu nhánh này sai
                board[hang][c] = 0
                cot[c] = False
                cheo_chinh[hang - c + n - 1] = False
                cheo_phu[hang + c] = False
        
        return False

    # Bắt đầu đệ quy từ hàng 0
    if Try(0):
        return board
    return None

## Thuật toán 2: Tham lam từng phần (Min-Conflicts Heuristic)

### Ý tưởng:
- Khởi tạo ngẫu nhiên, sau đó dời các con hậu bị tấn công đến ô ít bị tấn công nhất
- Thường dùng cho N rất lớn (N = 1000) khi Backtracking bị quá thời gian

In [2]:
import random

def tham_lam_tung_phan(n):
    """
    Giải bài toán N-Queens sử dụng heuristic Min-Conflicts.
    Hiệu quả với N lớn.
    """
    if n == 2 or n == 3: 
        return None
        
    # Khởi tạo ngẫu nhiên: queens[r] = c nghĩa là hàng r có hậu ở cột c
    queens = [i for i in range(n)]
    random.shuffle(queens) 
    
    def count_conflicts(r, c):
        """Đếm số xung đột nếu đặt hậu tại (r, c)"""
        conflicts = 0
        for i in range(n):
            if i == r: 
                continue
            # Kiểm tra cùng cột hoặc cùng đường chéo
            if queens[i] == c or abs(i - r) == abs(queens[i] - c):
                conflicts += 1
        return conflicts

    max_steps = 1000  # Giới hạn số lần lặp
    
    for _ in range(max_steps):
        # Tìm tất cả hậu bị xung đột
        conflicted_queens = [r for r in range(n) if count_conflicts(r, queens[r]) > 0]
        
        # Nếu không có xung đột, tìm thấy lời giải
        if not conflicted_queens:
            board = [[0] * n for _ in range(n)]
            for r in range(n):
                board[r][queens[r]] = 1
            return board
        
        # Chọn một hậu bị xung đột
        r = random.choice(conflicted_queens)
        
        # Tìm cột ít xung đột nhất cho hậu này
        min_c = queens[r]
        min_conflicts = count_conflicts(r, queens[r])
        
        for c in range(n):
            if c == queens[r]: 
                continue
            c_conflicts = count_conflicts(r, c)
            if c_conflicts < min_conflicts:
                min_conflicts = c_conflicts
                min_c = c
        
        # Di chuyển hậu đến cột mới
        queens[r] = min_c
        
    return None  # Không tìm thấy lời giải trong max_steps

## Ví dụ: N = 5

In [3]:
# Test với N = 5
n = 5
ket_qua = quay_lui(n)

if ket_qua is None:
    print(0)
else:
    print(1)
    for row in ket_qua:
        print(" ".join(map(str, row)))

1
1 0 0 0 0
0 0 1 0 0
0 0 0 0 1
0 1 0 0 0
0 0 0 1 0


## Ví dụ: N = 2 (Không có lời giải)

In [4]:
# Test với N = 2
n = 2
ket_qua = quay_lui(n)

if ket_qua is None:
    print(0)
else:
    print(1)
    for row in ket_qua:
        print(" ".join(map(str, row)))

0


## Chương trình chính (nhập từ bàn phím):

In [7]:
# Nhập N từ bàn phím
try:
    n = int(input("Nhập N: ").strip())
except (EOFError, ValueError):
    print(0)
else:
    # Gọi hàm thuật toán (mặc định dùng Backtracking để ra kết quả ổn định)
    ket_qua = quay_lui(n)
    # Để dùng Min-Conflicts, bỏ comment dòng dưới:
    # ket_qua = tham_lam_tung_phan(n)

    if ket_qua is None:
        print(0)
    else:
        print(1)
        for row in ket_qua:
            print(" ".join(map(str, row)))

1
1 0 0 0 0
0 0 1 0 0
0 0 0 0 1
0 1 0 0 0
0 0 0 1 0


## Giải thích chi tiết:

### Backtracking:
- **cot[c]**: Đánh dấu cột c đã có hậu
- **cheo_chinh[hang - c + n - 1]**: Đánh dấu đường chéo chính (từ trên trái xuống dưới phải)
- **cheo_phu[hang + c]**: Đánh dấu đường chéo phụ (từ trên phải xuống dưới trái)
- **Độ phức tạp**: O(N!) trong trường hợp xấu nhất

### Min-Conflicts:
- **Ưu điểm**: Nhanh hơn Backtracking cho N lớn (N > 100)
- **Nhược điểm**: Không đảm bảo tìm thấy lời giải (dùng ngẫu nhiên)
- **Độ phức tạp**: O(max_steps × N²)

### Trường hợp đặc biệt:
- **N = 1**: Có 1 lời giải (hậu ở (0,0))
- **N = 2, 3**: Không có lời giải
- **N ≥ 4**: Luôn có lời giải